## Data QC checker

This notebook checks the latest spreadsheet in `data/` for a few required-but-easy-to-miss data completeness issues. It does **not** fix anything -- it only reports which `resource_base` URLs are missing:

- a `definedTerms` row with `property == 'about'`
- a `definedTerms` row with `property == 'measurementTechnique'`
- any row at all in the `funding` sheet

Run this before (or alongside) `xlsx_batch_converter.ipynb` to catch these gaps early, since `xlsx_batch_converter.ipynb` raises a hard `KeyError` on a resource missing from `definedTerms` or `funding` rather than reporting it gracefully.

In [1]:
import os
import pandas as pd

In [2]:
script_path = os.getcwd()
parent_path = os.path.abspath(os.path.join(script_path, os.pardir))
data_path = os.path.join(script_path, 'data')

def get_latest_xlsx(data_path):
    """Return the most recently dated .xlsx file in data_path, ignoring
    Excel lock files (which start with '~$')."""
    files = [f for f in os.listdir(data_path) if f.lower().endswith('.xlsx') and not f.startswith('~$')]
    files.sort()
    return files[-1]

latest_file = get_latest_xlsx(data_path)
filepath = os.path.join(data_path, latest_file)
print(f'Checking: {latest_file}')

Checking: 2026_10_02_ProgramResourceCuration.xlsx


In [3]:
df_base = pd.read_excel(filepath, 'resource_base', engine='openpyxl')
df_definedTerms = pd.read_excel(filepath, 'definedTerms', engine='openpyxl')
df_funding = pd.read_excel(filepath, 'funding', engine='openpyxl')

In [4]:
def has_definedterm_property(df_dt, url, property_name):
    """True if the definedTerms sheet has at least one row for this url
    with the given property value."""
    matches = df_dt.loc[(df_dt['url'] == url) & (df_dt['property'] == property_name)]
    return len(matches) > 0

def has_funding_row(df_funding, url):
    """True if the funding sheet has at least one row for this url."""
    matches = df_funding.loc[df_funding['url'] == url]
    return len(matches) > 0

In [5]:
def build_qc_report(df_base, df_definedTerms, df_funding):
    issues = []
    for url in df_base['url'].dropna().unique().tolist():
        missing_about = not has_definedterm_property(df_definedTerms, url, 'about')
        missing_measurement_technique = not has_definedterm_property(df_definedTerms, url, 'measurementTechnique')
        missing_funding = not has_funding_row(df_funding, url)
        if missing_about or missing_measurement_technique or missing_funding:
            issues.append({
                'url': url,
                'missing_about': missing_about,
                'missing_measurementTechnique': missing_measurement_technique,
                'missing_funding': missing_funding,
            })
    return pd.DataFrame(issues, columns=['url', 'missing_about', 'missing_measurementTechnique', 'missing_funding'])

qc_report = build_qc_report(df_base, df_definedTerms, df_funding)

In [6]:
total_resources = df_base['url'].dropna().nunique()
print(f'Checked {total_resources} resources in {latest_file}')
print(f'Found {len(qc_report)} resource(s) with at least one issue:\n')

if len(qc_report) == 0:
    print('No issues found.')
else:
    for _, row in qc_report.iterrows():
        problems = []
        if row['missing_about']:
            problems.append("missing definedTerms 'about'")
        if row['missing_measurementTechnique']:
            problems.append("missing definedTerms 'measurementTechnique'")
        if row['missing_funding']:
            problems.append('missing funding')
        print(f"- {row['url']}: {', '.join(problems)}")

qc_report

Checked 28 resources in 2026_10_02_ProgramResourceCuration.xlsx
Found 0 resource(s) with at least one issue:

No issues found.


,url,missing_about,missing_measurementTechnique,missing_funding


## Fix measurementTechnique values using the ontology-priority mapping

`measurementTechnique` terms are sometimes curated using a lower-priority ontology (e.g. NCIT) even when a synonymous term exists in a higher-priority ontology. The functions below look up each `measurementTechnique` row's `prop.url` in the ordered mapping dict and replace it with its preferred-ontology equivalent, following the mapping chain until there's nothing left to replace -- a term can appear in more than one ontology, so a single lookup isn't always enough.

Once a `prop.url` is resolved, its preferred term label is looked up in that ontology's CSV export (`ontology_files/`) and used to update the `name` column too, so `name` and `prop.url` stay in sync. If no label can be found for a resolved url, `name` is left as-is rather than being blanked out.

`fix_measurement_technique()` writes the corrected `definedTerms` sheet back to the spreadsheet in place (it skips the write, but still returns the corrected data, if the workbook appears to be open in Excel). Run the preview cell first if you want to see what will change before committing to it.

Paths to the mapping file and ontology CSVs are resolved relative to `parent_path` (the `nde` folder one level up from this repo), assuming `measTechResearch/` is a sibling of `nde_meta_batch_converter/` -- adjust `MEAS_TECH_MAPPING_FILE` / `ONTOLOGY_FILES_DIR` below if that layout ever changes.

In [7]:
import json

MEAS_TECH_MAPPING_FILE = os.path.join(parent_path, 'measTechResearch', 'measTechNet', 'results', 'ordered_mapping_dict.json')

def load_meas_tech_mapping(mapping_file=MEAS_TECH_MAPPING_FILE):
    with open(mapping_file, encoding='utf-8') as infile:
        return json.load(infile)

def resolve_preferred_url(url, mapping):
    """Follow the mapping chain (lower-priority ontology term -> higher-
    priority ontology term) until the url is no longer a key in the
    mapping, i.e. until there's nothing left to replace. A url can be
    remapped more than once since a term may appear in multiple
    ontologies. Guards against an unexpected cycle."""
    seen = set()
    current = url
    while current in mapping and current not in seen:
        seen.add(current)
        current = mapping[current]
    return current

In [8]:
import re

ONTOLOGY_FILES_DIR = os.path.join(parent_path, 'measTechResearch', 'measTechAnalysis', 'data', 'ontology_files')

ONTOLOGY_PREFIX_TO_FILE = {
    'NCIT': 'NCIT.csv',
    'OBI': 'OBI.csv',
    'CHMO': 'CHMO.csv',
    'MMO': 'MMO.csv',
    'BAO': 'BAO.csv',
    'EFO': 'EFO.csv',
    'EDAM': 'EDAMT.csv',
}

def get_ontology_prefix(url):
    """Identify which ontology a term url belongs to, based on its domain
    and/or OBO-style id prefix (e.g. '..._obo/MMO_0000659' -> 'MMO')."""
    if 'edamontology.org' in url:
        return 'EDAM'
    if 'ebi.ac.uk/efo' in url:
        return 'EFO'
    if 'bioassayontology.org' in url:
        return 'BAO'
    match = re.search(r'obo/([A-Za-z]+)_', url)
    if match:
        return match.group(1).upper()
    return None

_ontology_label_cache = {}

def load_ontology_labels(prefix):
    """Load (and cache) a {Class ID: Preferred Label} lookup for one
    ontology, reading only the two columns needed. Returns {} if the
    ontology isn't one we have a CSV export for."""
    if prefix in _ontology_label_cache:
        return _ontology_label_cache[prefix]
    filename = ONTOLOGY_PREFIX_TO_FILE.get(prefix)
    if filename is None:
        _ontology_label_cache[prefix] = {}
        return {}
    onto_filepath = os.path.join(ONTOLOGY_FILES_DIR, filename)
    df_onto = pd.read_csv(onto_filepath, usecols=['Class ID', 'Preferred Label'])
    lookup = dict(zip(df_onto['Class ID'], df_onto['Preferred Label']))
    _ontology_label_cache[prefix] = lookup
    return lookup

def get_preferred_term_label(url):
    """Look up the Preferred Label for a term url in its ontology's CSV
    export. Returns None if the ontology or term isn't found."""
    prefix = get_ontology_prefix(url)
    if prefix is None:
        return None
    lookup = load_ontology_labels(prefix)
    return lookup.get(url)

In [9]:
def preview_measurement_technique_fixes(df_definedTerms, mapping):
    """Return the measurementTechnique rows whose prop.url would change,
    along with the preferred term's label (when found), without
    modifying df_definedTerms or touching the spreadsheet."""
    rows = []
    mt_rows = df_definedTerms.loc[df_definedTerms['property'] == 'measurementTechnique']
    for _, row in mt_rows.iterrows():
        resolved_url = resolve_preferred_url(row['prop.url'], mapping)
        if resolved_url != row['prop.url']:
            rows.append({
                'url': row['url'],
                'name': row['name'],
                'prop.url': row['prop.url'],
                'resolved_url': resolved_url,
                'resolved_name': get_preferred_term_label(resolved_url),
            })
    return pd.DataFrame(rows, columns=['url', 'name', 'prop.url', 'resolved_url', 'resolved_name'])

def write_definedterms_sheet(filepath, df_definedTerms_fixed):
    """Overwrite just the definedTerms sheet in filepath with the corrected
    dataframe, leaving every other sheet untouched."""
    with pd.ExcelWriter(filepath, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df_definedTerms_fixed.to_excel(writer, sheet_name='definedTerms', index=False)

def fix_measurement_technique(df_definedTerms, mapping, filepath):
    """Resolve every measurementTechnique row's prop.url to its highest-
    priority ontology equivalent (updating name to match wherever a
    preferred label can be found), then write the corrected definedTerms
    sheet back to filepath in place -- this actually modifies the
    spreadsheet on disk, it doesn't just return a preview.

    If the workbook appears to be open in Excel (its lock file is
    present), the write is skipped -- since writing to an open workbook
    can fail or corrupt the file -- but the corrected dataframe is still
    returned so you can inspect it and re-run once the file is closed."""
    df_fixed = df_definedTerms.copy()
    is_mt = df_fixed['property'] == 'measurementTechnique'
    changed_count = 0
    for idx in df_fixed.loc[is_mt].index:
        original_url = df_fixed.at[idx, 'prop.url']
        resolved_url = resolve_preferred_url(original_url, mapping)
        if resolved_url != original_url:
            df_fixed.at[idx, 'prop.url'] = resolved_url
            preferred_label = get_preferred_term_label(resolved_url)
            if preferred_label:
                df_fixed.at[idx, 'name'] = preferred_label
            changed_count += 1

    filename = os.path.basename(filepath)
    lock_file = os.path.join(os.path.dirname(filepath), f'~${filename}')
    if os.path.exists(lock_file):
        print(f'{filename} appears to be open in Excel (lock file present). Close it, then re-run fix_measurement_technique to write the fixes.')
        return df_fixed

    write_definedterms_sheet(filepath, df_fixed)
    print(f'Updated {changed_count} measurementTechnique url(s) in the definedTerms sheet of {filename}.')
    return df_fixed

In [10]:
meas_tech_mapping = load_meas_tech_mapping()
meas_tech_preview = preview_measurement_technique_fixes(df_definedTerms, meas_tech_mapping)
print(f'{len(meas_tech_preview)} measurementTechnique value(s) would be updated:')
meas_tech_preview

0 measurementTechnique value(s) would be updated:


,url,name,prop.url,resolved_url,resolved_name


In [11]:
## Run this cell to actually apply the fixes and overwrite the definedTerms
## sheet in filepath.
df_definedTerms_fixed = fix_measurement_technique(df_definedTerms, meas_tech_mapping, filepath)

Updated 0 measurementTechnique url(s) in the definedTerms sheet of 2026_10_02_ProgramResourceCuration.xlsx.
